> **Runs automatically.** This copy lives in the `football-picks` repository on GitHub, which runs it on a schedule and keeps the logs in the repository's `data` folder. Open it here to read or change it; the schedule picks up whatever is saved in the repository.

# College football game model v5: winners, spreads and totals, with preseason data and efficiency

This notebook replaces `ncaa_game_model_v4`. **The model is the same as versions 3 and 4**, and its picks still log as `cfb_margin_v3`. What's new:
- **Website data.** Step 12 saves this week's games, the season record and every graded game to `app_data/cfb.json` in your Google Drive. The picks website reads this file.
- **Projected scores show one decimal**, like "Memphis 28.3, Tulane 22.2", so they always agree with the spread.
- **Top picks inputs.** Step 12 also saves each game's chance to cover the spread, for the website's Top picks lists.

Version 4 added the plain-language game cards in Step 10.

Version 3 added **play-by-play efficiency**: how often each team's runs and passes succeed. After every game, the ratings learn from a blend of the real score and the points the teams' plays deserved, which filters out some of the luck in a final score.

**Where the data comes from.** The play-by-play comes from the same free sportsdataverse data as the schedules, so it doesn't use any of your CollegeFootballData calls. The efficiency file in your Drive at `cfbd/cfb_efficiency.parquet` is reused, and each run only downloads this season's play-by-play. Preseason data comes from CollegeFootballData through your `CFBD_API_KEY` secret.

**What it does**
1. Loads every game with an FBS team, plus betting lines.
2. Loads the preseason data, refreshing it when it's old.
3. Loads play-by-play efficiency for every team in every game since 2004.
4. Rates every team with Elo, tuned for college.
5. Gives every team a preseason rating for every season.
6. Rates every team's scoring offense and defense in points, learning from both real and deserved points.
7. Combines the ratings and home field into a model of the final margin.
8. Tests it on 2020–2025 against version 2, Elo alone and the betting market.
9. Builds and tests a totals model the same way.
10. Shows this week's games with a projected score for each.
11. Saves every model's picks to the college pick log and grades past picks.
12. Saves everything the website needs to one file.

**To run it:** in Colab, choose **Runtime → Run all**. It takes a minute or two (a few minutes if the efficiency file isn't in your Drive yet). Allow access to your secret and your Google Drive when Colab asks.

**Every week:** run it on **Tuesday**, before any evening games. By then the weekend's play-by-play is in (the files update on Sunday and Monday), and picks are only saved for games that haven't kicked off. Monday afternoon works too. Versions 3 and 4 log exactly the same picks, so running one by mistake does no harm, but only this version updates the website file. Run `game_review_v1` first, on the same day: it explains last week's results, and this notebook adds those explanations to the website.

In [ ]:
%pip install -q "polars>=1.0"

## Step 1: Load the games and betting lines

`games` has one row per game with at least one **FBS** team, the top division. Games against FCS and lower-division teams stay in, because they count toward each FBS team's rating.

**Betting lines** for past seasons come from many sportsbooks. For each game, the notebook takes the **median** of the sportsbooks' last lines: the spread, the total and both moneylines. It's a consensus line, much like the single line nflverse gives for the NFL. The 2020 data has no moneylines, and none of the data after 2019 has over/under odds.

**This week:** the past-lines file doesn't cover the current season, so the current lines come from ESPN. ESPN also fills in scores from the last few days that haven't reached the data files yet.

In [ ]:
import math
import time
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

import numpy as np
import polars as pl
import requests
from scipy.stats import norm

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)

DATA = "https://raw.githubusercontent.com/sportsdataverse/cfbfastR-data/main"
EASTERN = ZoneInfo("America/New_York")
now_utc = datetime.now(timezone.utc)
CURRENT_SEASON = now_utc.year if now_utc.month >= 7 else now_utc.year - 1

COLUMNS = {
    "game_id": pl.Int64, "season": pl.Int32, "week": pl.Int32, "season_type": pl.String, "start_date": pl.String,
    "neutral_site": pl.Boolean, "conference_game": pl.Boolean,
    "home_id": pl.Int64, "home_team": pl.String, "home_division": pl.String, "home_conference": pl.String, "home_points": pl.Int32,
    "away_id": pl.Int64, "away_team": pl.String, "away_division": pl.String, "away_conference": pl.String, "away_points": pl.Int32,
}

# Every game since 2001, one file per season
frames = []
for season in range(2001, CURRENT_SEASON + 1):
    try:
        s = pl.read_parquet(f"{DATA}/schedules/parquet/cfb_schedules_{season}.parquet")
    except Exception:
        print(f"No schedule file for {season} yet.")
        continue
    frames.append(s.select([pl.col(c).cast(t) for c, t in COLUMNS.items()]))
schedule = pl.concat(frames)

# Keep games with at least one FBS team (the top division)
fbs = (
    pl.concat([schedule.select("season", team="home_id", division="home_division"),
               schedule.select("season", team="away_id", division="away_division")])
    .filter(pl.col("division") == "fbs").select("season", "team").unique()
)
fbs_by_season = {}
for season, team in fbs.iter_rows():
    fbs_by_season.setdefault(season, set()).add(team)
games = (
    schedule
    .join(fbs.with_columns(home_fbs=pl.lit(True)).rename({"team": "home_id"}), on=["season", "home_id"], how="left")
    .join(fbs.with_columns(away_fbs=pl.lit(True)).rename({"team": "away_id"}), on=["season", "away_id"], how="left")
    .with_columns(pl.col("home_fbs").fill_null(False), pl.col("away_fbs").fill_null(False))
    .filter(pl.col("home_fbs") | pl.col("away_fbs"))
    .with_columns(kickoff=pl.col("start_date").str.to_datetime("%Y-%m-%dT%H:%M:%S%.fZ", time_zone="UTC"))
)
print(f"{games.height:,} games with an FBS team, seasons {games['season'].min()}–{games['season'].max()}")

In [ ]:
def american_to_prob(odds):
    return pl.when(odds < 0).then(-odds / (-odds + 100)).otherwise(100 / (odds + 100))


def market_lines(odds, games, rounds=3):
    """Consensus lines for past games: the median across sportsbooks of each game's last spread, total and moneylines.

    The file labels teams with each sportsbook's own abbreviations, so this first works out which row is the
    home team. It starts from team names it recognizes, then learns each abbreviation from games where the
    other team's row was recognized."""
    sides = (
        odds.filter(pl.col("market_type").is_in(["spread", "money_line"]))
        .join(games.select("game_id", "home_id", "away_id"), on="game_id")
        .with_columns(rows=pl.len().over("game_id", "book", "date_time", "market_type"))
        .filter(pl.col("rows") == 2)                                    # one row per team
        .with_columns(pair=pl.struct("game_id", "book", "date_time", "market_type").hash())
    )
    names = pl.concat([games.select(team="home_id", abbr="home_team"), games.select(team="away_id", abbr="away_team")]).unique()
    for _ in range(rounds):
        known = (
            sides.join(names.rename({"team": "home_id"}).with_columns(is_home=pl.lit(True)), on=["home_id", "abbr"], how="left")
            .join(names.rename({"team": "away_id"}).with_columns(is_away=pl.lit(True)), on=["away_id", "abbr"], how="left")
            .with_columns(side=pl.when(pl.col("is_home") & pl.col("is_away").is_null()).then(1)
                          .when(pl.col("is_away") & pl.col("is_home").is_null()).then(-1))
        )
        # If one team's row is recognized, the other row is the other team
        known = known.with_columns(side=pl.coalesce("side", -pl.col("side").drop_nulls().first().over("pair")))
        learned = (
            known.filter(pl.col("side").is_not_null())
            .select(team=pl.when(pl.col("side") == 1).then(pl.col("home_id")).otherwise(pl.col("away_id")), abbr=pl.col("abbr"))
            .group_by("team", "abbr").len().filter(pl.col("len") >= 3).drop("len")
        )
        names = pl.concat([names, learned]).unique()
    known = known.filter(pl.col("side").is_not_null())
    spread = known.filter(pl.col("market_type") == "spread").group_by("game_id").agg(
        spread_line=(-pl.col("lines") * pl.col("side")).median())       # positive = home team favored
    moneyline = known.filter(pl.col("market_type") == "money_line").group_by("game_id").agg(
        home_moneyline=pl.col("odds").filter(pl.col("side") == 1).median(),
        away_moneyline=pl.col("odds").filter(pl.col("side") == -1).median(),
    )
    totals = odds.filter((pl.col("market_type") == "total") & pl.col("lines").is_not_null()).group_by("game_id").agg(
        total_line=pl.col("lines").median(),
        over_odds=pl.col("odds").filter(pl.col("abbr") == "over").median(),
        under_odds=pl.col("odds").filter(pl.col("abbr") == "under").median(),
    )
    return spread.join(totals, on="game_id", how="full", coalesce=True).join(moneyline, on="game_id", how="full", coalesce=True)


ESPN = "https://site.api.espn.com/apis/site/v2/sports/football/college-football/scoreboard"


def american(text):
    """'+150' -> 150, 'EVEN' -> 100, anything else -> None."""
    text = str(text or "").strip().upper()
    if text == "EVEN":
        return 100
    try:
        return int(text.replace("+", ""))
    except ValueError:
        return None


def espn_week(season, week, season_type):
    """This week's scores and sportsbook lines from ESPN, for games involving FBS teams."""
    params = {"groups": 80, "dates": season, "week": week, "seasontype": 3 if season_type == "postseason" else 2, "limit": 400}
    for attempt in range(3):
        try:
            response = requests.get(ESPN, params=params, timeout=30)
            response.raise_for_status()
            events = response.json().get("events", [])
            break
        except (requests.RequestException, ValueError):
            time.sleep(3)
    else:
        print(f"Couldn't reach ESPN for week {week}. Lines for that week will be missing; try again later.")
        return []
    rows = []
    for e in events:
        c = e["competitions"][0]
        teams = {t["homeAway"]: t for t in c["competitors"]}
        odds = (c.get("odds") or [{}])[0]
        moneyline = odds.get("moneyline") or {}
        total = odds.get("total") or {}
        final = c["status"]["type"].get("completed", False)
        rows.append({
            "game_id": int(e["id"]),
            "espn_home_points": int(teams["home"]["score"]) if final else None,
            "espn_away_points": int(teams["away"]["score"]) if final else None,
            "espn_spread": -float(odds["spread"]) if odds.get("spread") is not None else None,   # ESPN gives the home team's line
            "espn_total": float(odds["overUnder"]) if odds.get("overUnder") is not None else None,
            "espn_home_ml": american((moneyline.get("home") or {}).get("close", {}).get("odds")),
            "espn_away_ml": american((moneyline.get("away") or {}).get("close", {}).get("odds")),
            "espn_over_odds": american((total.get("over") or {}).get("close", {}).get("odds")),
            "espn_under_odds": american((total.get("under") or {}).get("close", {}).get("odds")),
        })
    return rows


# Past seasons' lines (2006 on)
odds = pl.read_parquet(f"{DATA}/betting/parquet/cfb_line_odds.parquet").with_columns(pl.col("game_id").cast(pl.Int64))
games = games.join(market_lines(odds, games), on="game_id", how="left")

# This season's recent and upcoming weeks: scores that haven't reached the data files yet, and current lines
recent = games.filter(
    (pl.col("season") == CURRENT_SEASON) & pl.col("home_points").is_null()
    & pl.col("kickoff").is_between(now_utc - timedelta(days=10), now_utc + timedelta(days=8))
).select("week", "season_type").unique()
espn_rows = [row for w, st in recent.iter_rows() for row in espn_week(CURRENT_SEASON, w, st)]
espn_schema = {"game_id": pl.Int64, "espn_home_points": pl.Int32, "espn_away_points": pl.Int32, "espn_spread": pl.Float64,
               "espn_total": pl.Float64, "espn_home_ml": pl.Int64, "espn_away_ml": pl.Int64,
               "espn_over_odds": pl.Int64, "espn_under_odds": pl.Int64}
espn = pl.DataFrame(espn_rows, schema=espn_schema).unique("game_id")
games = (
    games.join(espn, on="game_id", how="left")
    .with_columns(
        home_points=pl.coalesce("home_points", "espn_home_points"),
        away_points=pl.coalesce("away_points", "espn_away_points"),
        spread_line=pl.coalesce("spread_line", "espn_spread"),
        total_line=pl.coalesce("total_line", "espn_total"),
        home_moneyline=pl.coalesce("home_moneyline", "espn_home_ml"),
        away_moneyline=pl.coalesce("away_moneyline", "espn_away_ml"),
        over_odds=pl.coalesce("over_odds", "espn_over_odds"),
        under_odds=pl.coalesce("under_odds", "espn_under_odds"),
    )
    .drop([c for c in espn_schema if c != "game_id"])
    .with_columns(
        result=pl.col("home_points") - pl.col("away_points"),
        total=pl.col("home_points") + pl.col("away_points"),
        imp_home=american_to_prob(pl.col("home_moneyline")),
        imp_away=american_to_prob(pl.col("away_moneyline")),
        imp_over=american_to_prob(pl.col("over_odds")),
        imp_under=american_to_prob(pl.col("under_odds")),
    )
    .with_columns(
        mkt_p_home=pl.col("imp_home") / (pl.col("imp_home") + pl.col("imp_away")),
        mkt_p_over=pl.col("imp_over") / (pl.col("imp_over") + pl.col("imp_under")),
    )
    .sort("kickoff", "game_id")
)

played = games.filter(pl.col("result").is_not_null())
print(f"Latest result: {played['kickoff'].max().astimezone(EASTERN):%a %b %d, %Y}. "
      f"Games with a betting line: {games['spread_line'].is_not_null().sum():,}. "
      f"Lines from ESPN this week: {espn['espn_spread'].is_not_null().sum()}.")

## Step 2: Preseason data from CollegeFootballData

For every FBS team and season, the saved file has:
- **Recruiting**: 247Sports points for each recruiting class, since 2002.
- **Talent**: the 247Sports talent composite for the whole roster, since 2015.
- **Returning production**: the share of last season's production (by expected points added) that's back, since 2014.
- **Transfers**: how many players transferred in and out through the portal, and how highly rated the incoming ones were, since 2021.

This cell loads the file from your Drive. If your key is available and the file is more than two weeks old, it refreshes this season's rows first. If anything goes wrong with CollegeFootballData, it says so and keeps going with the saved file.

In [ ]:
import os
from pathlib import Path

REFRESH_DAYS = 14   # refresh this season's preseason data when the saved file is older than this

# The saved preseason file lives in your Google Drive, in the cfbd folder
try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    CFBD_FOLDER = Path("/content/drive/MyDrive/cfbd")
except ImportError:
    userdata = None
    CFBD_FOLDER = Path("cfbd")
CFBD_FILE = CFBD_FOLDER / "cfbd_preseason.parquet"

# Your key comes from Colab's Secrets panel, or from the repository secret on GitHub, so it never appears in the notebook
CFBD_KEY = None
if userdata is not None:
    try:
        CFBD_KEY = userdata.get("CFBD_API_KEY")
    except Exception:
        print("No CFBD_API_KEY secret this notebook can use, so it will rely on the saved preseason file.")
else:
    CFBD_KEY = os.environ.get("CFBD_API_KEY") or None      # GitHub Actions passes the repository secret this way

API = "https://api.collegefootballdata.com"
session = requests.Session()
session.headers.update({"Authorization": f"Bearer {(CFBD_KEY or '').strip()}", "Accept": "application/json"})
calls_used = 0


def cfbd(path, **params):
    """One request to CollegeFootballData. Each request uses one of your monthly calls."""
    global calls_used
    for attempt in range(4):
        response = session.get(f"{API}{path}", params=params, timeout=60)
        calls_used += 1
        if response.status_code == 401:
            raise RuntimeError("CollegeFootballData didn't accept the key. Check the value of the CFBD_API_KEY secret.")
        if response.status_code == 429:                  # too many requests too fast: wait and try again
            time.sleep(10 * (attempt + 1))
            continue
        response.raise_for_status()
        time.sleep(0.3)
        return response.json()
    raise RuntimeError(f"CollegeFootballData kept saying 'too many requests' for {path}. Wait a few minutes and try again.")


def table(rows, columns):
    """Rows from the API as a table with the columns we keep (missing columns become empty)."""
    df = pl.DataFrame(rows, infer_schema_length=None) if rows else pl.DataFrame()
    return df.select([pl.col(src).alias(dst) if src in df.columns else pl.lit(None).alias(dst) for src, dst in columns.items()])


def download_preseason(years):
    """Recruiting, talent, returning production and transfers for the given years: one call per year for each."""
    def by_year(path, first_year):
        rows = []
        for year in years:
            if year >= first_year:
                rows.extend(cfbd(path, year=year) or [])
        return rows

    rec = table(by_year("/recruiting/teams", 2002), {"year": "season", "team": "team", "rank": "recruit_rank", "points": "recruit_points"})
    tal = table(by_year("/talent", 2015), {"year": "season", "team": "team", "talent": "talent"})
    ret = table(by_year("/player/returning", 2005), {
        "season": "season", "team": "team", "percentPPA": "returning_ppa",
        "percentPassingPPA": "returning_passing_ppa", "percentRushingPPA": "returning_rushing_ppa",
        "percentReceivingPPA": "returning_receiving_ppa", "usage": "returning_usage"})
    trn = table(by_year("/player/portal", 2021), {"season": "season", "origin": "origin", "destination": "destination", "rating": "rating"})
    transfers_in = trn.filter(pl.col("destination").is_not_null()).group_by("season", team="destination").agg(
        transfers_in=pl.len(), transfers_in_rating=pl.col("rating").cast(pl.Float64).mean())
    transfers_out = trn.filter(pl.col("origin").is_not_null()).group_by("season", team="origin").agg(
        transfers_out=pl.len(), transfers_out_rating=pl.col("rating").cast(pl.Float64).mean())
    parts = [d.with_columns(pl.col("season").cast(pl.Int32), pl.col("team").cast(pl.String))
             for d in (rec, tal, ret, transfers_in, transfers_out)]
    out = pl.concat([d.select("season", "team") for d in parts]).drop_nulls().unique()
    for d in parts:
        out = out.join(d, on=["season", "team"], how="left")
    return out.sort("season", "team")


preseason = pl.read_parquet(CFBD_FILE) if CFBD_FILE.exists() else None
stale = preseason is not None and (time.time() - CFBD_FILE.stat().st_mtime) > REFRESH_DAYS * 86400
try:
    if CFBD_KEY and preseason is None:
        print("Downloading preseason data from CollegeFootballData for the first time (about 70 calls)...")
        preseason = download_preseason(range(2002, now_utc.year + 1))
    elif CFBD_KEY and (stale or CURRENT_SEASON not in preseason["season"].to_list()):
        print(f"Refreshing {CURRENT_SEASON} preseason data from CollegeFootballData (4 calls)...")
        fresh = download_preseason([CURRENT_SEASON])
        preseason = pl.concat([preseason.filter(pl.col("season") != CURRENT_SEASON),
                               fresh.select(preseason.columns).cast(preseason.schema)])
except Exception as error:
    calls_used = 0
    print(f"Couldn't update the preseason data ({error}). Using the saved file instead, if there is one.")
if CFBD_KEY and calls_used and preseason is not None:
    CFBD_FOLDER.mkdir(parents=True, exist_ok=True)
    preseason.write_parquet(CFBD_FILE)
    preseason.write_csv(CFBD_FOLDER / "cfbd_preseason.csv")
    print(f"Saved to {CFBD_FILE}. This used {calls_used} CollegeFootballData calls.")

if preseason is None:
    print("No preseason data, so this run works like version 1. Add the CFBD_API_KEY secret to turn it on.")
else:
    print(f"Preseason data: {preseason.height:,} team-seasons, {preseason['season'].min()}–{preseason['season'].max()}.")

## Step 3: Play-by-play efficiency

For every team in every game, this counts its **runs and passes** (sacks count as passes) and how many were **successful**. A play is successful if it gains at least half the yards to go on 1st down, 70% of them on 2nd down, or all of them on 3rd or 4th down. It's the standard success rate in college football analytics.

Success rate shows how well a team really played better than the final score does. A team can lose a game it controlled because of fumbles, one fluky long play or a missed field goal. In testing it also worked better than expected points added (EPA), which swings with turnovers and big plays almost as much as the score does.

The play-by-play comes from two sets of free files: older ones for 2004–2013, and sportsdataverse's newer ones from 2014 on, built from CollegeFootballData's play-by-play. The first run downloads all of them, about 1.5 GB, and saves a small summary to your Drive. After that, each run only downloads this season's file. Games that aren't in the files yet count with their final scores only.

In [ ]:
import io

PBP_FILES = {
    "older": f"{DATA}/pbp/parquet/play_by_play_{{}}.parquet",     # 2004–2013, in the same repo as the schedules
    "newer": "https://github.com/sportsdataverse/sportsdataverse-data/releases/download/cfbfastR_cfb_pbp/play_by_play_{}.parquet",
}
FIRST_PBP_SEASON, NEWER_FROM = 2004, 2014
EFFICIENCY_FILE = CFBD_FOLDER / "cfb_efficiency.parquet"
EFFICIENCY_SCHEMA = {"season": pl.Int32, "game_id": pl.Int64, "team_id": pl.Int64, "plays": pl.Int64, "successes": pl.Int64}


def fetch(url):
    """Download one file, trying up to 3 times. Returns None if the file doesn't exist (yet)."""
    for attempt in range(3):
        try:
            response = requests.get(url, timeout=300)
            if response.status_code == 404:
                return None
            response.raise_for_status()
            return response.content
        except requests.RequestException:
            time.sleep(3 * (attempt + 1))
    raise RuntimeError(f"couldn't download {url.split('/')[-1]}")


def season_efficiency(season):
    """One row per team per game: its offensive plays (runs and passes, sacks included) and how many were successful."""
    newer = season >= NEWER_FROM
    content = fetch(PBP_FILES["newer" if newer else "older"].format(season))
    if content is None:
        return None
    if newer:
        p = pl.read_parquet(io.BytesIO(content), columns=["game_id", "pos_team", "home_team", "home_team_id", "away_team_id",
                                                          "rush", "pass", "down", "distance", "yards_gained", "EPA"])
        p = p.select("game_id", "rush", "pass", "down", "distance", "EPA", yards="yards_gained",
                     team_id=pl.when(pl.col("pos_team") == pl.col("home_team")).then(pl.col("home_team_id")).otherwise(pl.col("away_team_id")))
    else:   # the older files use different column names, and team ids instead of names
        p = pl.read_parquet(io.BytesIO(content), columns=["game_id", "pos_team", "rush", "pass", "start.down", "start.distance", "statYardage", "EPA"])
        p = p.select("game_id", "rush", "pass", "EPA", team_id="pos_team", down="start.down", distance="start.distance", yards="statYardage")
    del content
    p = p.with_columns(pl.col(c).cast(pl.Float64) for c in ["rush", "pass", "down", "distance", "yards"]).filter(
        ((pl.col("rush") == 1) | (pl.col("pass") == 1)) & pl.col("EPA").is_not_null())     # runs and passes with complete data
    # A play is successful if it gains half the yards to go on 1st down, 70% of them on 2nd, and all of them on 3rd or 4th
    needed = pl.when(pl.col("down") == 1).then(0.5).when(pl.col("down") == 2).then(0.7).otherwise(1.0) * pl.col("distance")
    return (
        p.with_columns(success=(pl.col("yards") >= needed).fill_null(False))
        .group_by(pl.col("game_id").cast(pl.Int64), pl.col("team_id").cast(pl.Int64))
        .agg(plays=pl.len(), successes=pl.col("success").sum())
        .with_columns(season=pl.lit(season))
        .select(list(EFFICIENCY_SCHEMA)).cast(EFFICIENCY_SCHEMA)
    )


# Load the saved file, then download any missing seasons plus this season, which changes every week
efficiency = pl.read_parquet(EFFICIENCY_FILE).cast(EFFICIENCY_SCHEMA) if EFFICIENCY_FILE.exists() else pl.DataFrame(schema=EFFICIENCY_SCHEMA)
saved = set(efficiency["season"].unique().to_list())
todo = [s for s in range(FIRST_PBP_SEASON, CURRENT_SEASON + 1) if s not in saved or s == CURRENT_SEASON]
if len(todo) > 1:
    print(f"Building the efficiency file from play-by-play, {len(todo)} seasons. This happens once and takes a few minutes.")
updated, failed, problem = [], [], None
for i, season in enumerate(todo):
    try:
        part = season_efficiency(season)
    except Exception as error:
        failed, problem = todo[i:], error           # stop at the first failure; the next run tries again
        break
    if part is not None and part.height:
        efficiency = pl.concat([efficiency.filter(pl.col("season") != season), part])
        updated.append(season)
    if len(todo) > 1:
        print(season, end=" ", flush=True)
if len(todo) > 1 and (updated or failed != todo):
    print()
if updated:
    efficiency = efficiency.sort("season", "game_id", "team_id")
    try:
        CFBD_FOLDER.mkdir(parents=True, exist_ok=True)
        efficiency.write_parquet(EFFICIENCY_FILE)
    except OSError as error:
        print(f"Couldn't save {EFFICIENCY_FILE} ({error}). This run still uses the new data.")
if failed:
    print(f"Couldn't download the play-by-play ({problem}). The next run will try again.")
    lost = [s for s in failed if s not in saved]
    if lost:
        seasons = f"{lost[0]}–{lost[-1]}" if len(lost) > 1 else f"{lost[0]}"
        print(f"  No efficiency data for {seasons} this run, so those games count with their final scores only.")
    if CURRENT_SEASON in failed and CURRENT_SEASON in saved:
        print("  This season's efficiency comes from the saved file, so it may be missing the latest games.")

if efficiency.is_empty():
    print("No play-by-play data, so this run works like version 2.")
else:
    print(f"Efficiency data: {efficiency['game_id'].n_unique():,} games, {efficiency['season'].min()}–{efficiency['season'].max()}, "
          f"saved at {EFFICIENCY_FILE}.")
    # Recent FBS games that aren't in the play-by-play files yet
    both_teams = efficiency.group_by("game_id").len().filter(pl.col("len") == 2)["game_id"].to_list()
    recent = games.filter(pl.col("result").is_not_null() & pl.col("home_fbs") & pl.col("away_fbs")
                          & (pl.col("kickoff") > now_utc - timedelta(days=8)))
    missing = recent.filter(~pl.col("game_id").is_in(both_teams))
    if missing.height:
        print(f"{missing.height} of the {recent.height} FBS games from the last 8 days aren't in the play-by-play files yet, so they count "
              "with their final scores only. The files usually update on Sunday and Monday.")
    elif recent.height:
        print(f"All {recent.height} FBS games from the last 8 days are in.")

## Step 4: Elo ratings, tuned for college

This is the same Elo model as versions 1 and 2. The winner takes points from the loser after each game, and each game is predicted *before* its result updates the ratings.
- **Lower-division teams start far below FBS teams.** A team from FCS or below starts 450 points lower, about 25 points of margin.
- **Programs stay good or bad for years.** Each offseason, a team moves halfway toward a target that's mostly (75%) its own average rating over the last 8 seasons.

Home field is worth 60 Elo points, and about 18 Elo points equal 1 point of margin. Starting Elo from the preseason ratings too made no difference on the practice seasons, so Elo is unchanged.

In [ ]:
ELO_SETTINGS = {"k": 40, "home_field": 60, "regress": 0.5, "non_fbs_gap": 450,
                "program_weight": 0.75, "program_years": 8}     # chosen on 2016–2019 practice seasons
ELO_PER_POINT = 18   # in college, about 18 Elo points = 1 point of margin (fit on 2006–2015)


def run_elo(games, k=40, home_field=60, regress=0.5, non_fbs_gap=450, program_weight=0.75, program_years=8):
    """Predict every game in date order, then update ratings once it's played."""
    ratings, history, season, rows = {}, {}, None, []

    def start(team, season):
        return 1500 if team in fbs_by_season.get(season, ()) else 1500 - non_fbs_gap

    for g in games.iter_rows(named=True):
        if g["season"] != season:
            if season is not None:
                # Offseason: pull each team partway toward its program's usual level
                for team, r in ratings.items():
                    history.setdefault(team, []).append(r)
                for team, r in ratings.items():
                    program = np.mean(history[team][-program_years:])
                    target = program_weight * program + (1 - program_weight) * start(team, g["season"])
                    ratings[team] = target + (1 - regress) * (r - target)
            season = g["season"]

        home, away = g["home_id"], g["away_id"]
        r_home = ratings.get(home, start(home, season))
        r_away = ratings.get(away, start(away, season))
        edge = 0 if g["neutral_site"] else home_field
        diff = r_home + edge - r_away
        p_home = 1 / (1 + 10 ** (-diff / 400))                 # the prediction, made BEFORE the game
        rows.append({"game_id": g["game_id"], "elo_home": r_home, "elo_away": r_away,
                     "elo_p_home": p_home, "elo_spread": diff / ELO_PER_POINT})

        ratings.setdefault(home, r_home)
        ratings.setdefault(away, r_away)
        if g["result"] is None:
            continue
        margin = g["result"]
        actual = 1.0 if margin > 0 else 0.0 if margin < 0 else 0.5
        winner_diff = diff if margin > 0 else -diff
        mov_mult = math.log(abs(margin) + 1) * 2.2 / (winner_diff * 0.001 + 2.2)
        shift = k * mov_mult * (actual - p_home)
        ratings[home] = r_home + shift
        ratings[away] = r_away - shift
    return pl.DataFrame(rows), ratings


elo_preds, elo_ratings = run_elo(games, **ELO_SETTINGS)
df = games.join(elo_preds, on="game_id")

team_names = dict(pl.concat([games.select("home_id", "home_team"), games.select(pl.col("away_id").alias("home_id"), pl.col("away_team").alias("home_team"))]).iter_rows())
top = sorted(((r, team_names.get(t, t)) for t, r in elo_ratings.items() if t in fbs_by_season.get(CURRENT_SEASON, ())), reverse=True)[:10]
print("Top 10 FBS teams by Elo right now:")
for i, (r, name) in enumerate(top, 1):
    print(f"{i:>2}. {name:<20} {r:.0f}")

## Step 5: Preseason ratings

**Season strength.** First, the notebook measures how good every team actually was in each past season: the margin it would beat an average FBS team by, solved from all of that season's scores at once. Sports sites publish ratings like this as SRS.

**Preseason rating.** Then it predicts each team's season strength from what's known before the season starts:

| Input | What it means |
|---|---|
| `last_season` | Last season's strength |
| `program` | Average strength over the last 5 seasons |
| `recruiting` | Average points of the last 4 recruiting classes, roughly this year's roster |
| `returning` | How much of last season's production is back, compared with a typical team |
| `transfers_in`, `transfers_out`, `transfers_in_quality` | Portal moves, and how highly rated the incoming transfers are |

A regression learns how much each one matters. **Each season's preseason ratings come from a regression fit only on earlier seasons**, so a test season's ratings never use anything from that season.

On 2020–2025, the preseason rating missed each team's actual season strength by about 5.7 points. Using last season alone missed by 6.0, and adding the 5-season average got it to 5.85. Talent didn't help once recruiting was in, so it's saved but not used.

In [ ]:
PRIOR_FEATURES = ["last_season", "program", "recruiting", "returning", "transfers_in", "transfers_out", "transfers_in_quality"]
NON_FBS_STRENGTH = -12.0   # a typical lower-division team, in points vs. an average FBS team


def season_strength(games, ridge=2.0):
    """How good each team was over a whole season, in points vs. an average FBS team.
    It's the margin each team would beat an average team by, solved from all of that season's scores at once."""
    rows = []
    played = games.filter(pl.col("result").is_not_null())
    for season in sorted(played["season"].unique()):
        g = played.filter(pl.col("season") == season)
        teams = sorted(set(g["home_id"]) | set(g["away_id"]))
        idx = {t: i for i, t in enumerate(teams)}
        X = np.zeros((g.height, len(teams) + 1))
        for r, (h, a, neutral) in enumerate(g.select("home_id", "away_id", "neutral_site").iter_rows()):
            X[r, idx[h]], X[r, idx[a]], X[r, -1] = 1, -1, 0 if neutral else 1
        penalty = ridge * np.eye(len(teams) + 1)
        penalty[-1, -1] = 0                                  # don't shrink home field
        coef = np.linalg.solve(X.T @ X + penalty, X.T @ g["result"].to_numpy().astype(float))
        fbs_teams = fbs_by_season.get(season, set())
        fbs_mean = np.mean([coef[idx[t]] for t in teams if t in fbs_teams])
        rows += [{"season": season, "team_id": t, "strength": coef[idx[t]] - fbs_mean} for t in teams]
    return pl.DataFrame(rows).with_columns(pl.col("season").cast(pl.Int32))


strength = season_strength(games)

# One row per FBS team and season, with everything known before that season's first game
names = pl.concat([games.select("season", team_id="home_id", team="home_team"),
                   games.select("season", team_id="away_id", team="away_team")]).unique(["season", "team_id"])
fbs_rows = pl.DataFrame([{"season": s, "team_id": t} for s, ts in fbs_by_season.items() for t in ts],
                        schema={"season": pl.Int32, "team_id": pl.Int64})
past = pl.concat([strength.select(season=pl.col("season") + k, team_id="team_id", s="strength", k=pl.lit(k)) for k in range(1, 6)])
pre_table = (
    fbs_rows.join(names, on=["season", "team_id"], how="left")
    .join(strength, on=["season", "team_id"], how="left")
    .join(past.filter(pl.col("k") == 1).select("season", "team_id", last_season="s"), on=["season", "team_id"], how="left")
    .join(past.group_by("season", "team_id").agg(program=pl.col("s").mean()), on=["season", "team_id"], how="left")
)
if preseason is not None:
    p = preseason.with_columns(pl.col("season").cast(pl.Int32))
    recruiting4 = pl.concat([p.select(season=pl.col("season") + k, team="team", points="recruit_points") for k in range(4)]) \
        .group_by("season", "team").agg(recruiting=pl.col("points").mean())        # this year's class and the 3 before it
    pre_table = pre_table.join(recruiting4, on=["season", "team"], how="left").join(
        p.select("season", "team", "returning_ppa",
                 pl.col("transfers_in").cast(pl.Float64), pl.col("transfers_out").cast(pl.Float64), "transfers_in_rating"),
        on=["season", "team"], how="left")
else:
    pre_table = pre_table.with_columns(recruiting=pl.lit(None, pl.Float64), returning_ppa=pl.lit(None, pl.Float64),
                                       transfers_in=pl.lit(None, pl.Float64), transfers_out=pl.lit(None, pl.Float64),
                                       transfers_in_rating=pl.lit(None, pl.Float64))
pre_table = pre_table.with_columns(
    last_season=pl.col("last_season").fill_null(-15.0),                 # new to FBS: start well below average
    program=pl.coalesce("program", "last_season").fill_null(-15.0),
    recruiting=pl.col("recruiting").fill_null(80.0),                     # no ranked classes: a weak recruiting team
    returning=(pl.col("returning_ppa") - pl.col("returning_ppa").mean()).fill_null(0.0),
    transfers_in_quality=(pl.col("transfers_in") * (pl.col("transfers_in_rating") - 0.85)).fill_null(0.0),
    transfers_in=pl.col("transfers_in").fill_null(0.0),
    transfers_out=pl.col("transfers_out").fill_null(0.0),
).sort("season", "team_id")


def walk_forward_priors(table, features=PRIOR_FEATURES, first=2002):
    """Preseason rating for every team-season, from a regression fit only on earlier seasons."""
    out = []
    for season in sorted(table["season"].unique()):
        train = table.filter(pl.col("season").is_between(first, season - 1) & pl.col("strength").is_not_null())
        if train["season"].n_unique() < 3:
            continue
        X = np.column_stack([train.select(features).to_numpy(), np.ones(train.height)])
        w, *_ = np.linalg.lstsq(X, train["strength"].to_numpy(), rcond=None)
        rows = table.filter(pl.col("season") == season)
        pred = np.column_stack([rows.select(features).to_numpy(), np.ones(rows.height)]) @ w
        out.append(rows.select("season", "team_id", "team").with_columns(preseason_rating=pl.Series(pred)))
    return pl.concat(out), w


priors, prior_weights = walk_forward_priors(pre_table)
prior_by_team = {(s, t): r for s, t, r in priors.select("season", "team_id", "preseason_rating").iter_rows()}
if preseason is None:
    prior_by_team = {}          # without CollegeFootballData, version 2 works exactly like version 1
    print("No preseason data, so there are no preseason ratings this run.")
else:
    print(f"This season's preseason rating, from {', '.join(PRIOR_FEATURES)}. What each is worth (latest fit):")
    for name, weight in zip(PRIOR_FEATURES + ["constant"], prior_weights):
        print(f"  {name:<22} {weight:+.3f}")
    print("\nTop 10 preseason ratings this season (points better than an average FBS team):")
    display(priors.filter(pl.col("season") == CURRENT_SEASON).join(pre_table.select("season", "team_id", "last_season", "recruiting", "returning_ppa"),
            on=["season", "team_id"]).sort("preseason_rating", descending=True).head(10).select(
        "team", preseason=pl.col("preseason_rating").round(1), last_season=pl.col("last_season").round(1),
        recruiting=pl.col("recruiting").round(1), returning_production=pl.col("returning_ppa").round(2)))

## Step 6: Points ratings, learning from real and deserved points

Points ratings track each team's **scoring offense and defense**, in points compared with the league average. From them, every game gets an **expected score for each team**: the difference is an expected margin (Step 7), and the sum is an expected total (Step 9). As in version 2, each season starts with every team pulled 80% of the way toward its preseason rating.

**New in version 3: deserved points.** A regression learns how many points usually go with a team's plays and successful plays in a game, separately for the older and newer play-by-play files. After each game, the ratings move toward a **50/50 blend of the real score and the deserved points**. Games without play-by-play use the real score only.

On the 2016–2019 practice seasons, this cut log loss from 0.468 to 0.464 and the spread miss from 13.03 to 12.92 points. Leaning harder on deserved points did worse than the 50/50 blend, and so did versions built on EPA.

Version 2's ratings, from real scores only, are kept for the comparison in Step 8 and for the totals model, because deserved points didn't make totals any better.

In [ ]:
POINTS_SETTINGS = {"k": 0.15, "regress": 0.05, "home_field": 0.5, "non_fbs_gap": 12,
                   "start_avg": 27.0, "avg_rate": 0.002, "prior_pull": 0.8, "blend": 0.5}     # chosen on 2016–2019 practice seasons
MIN_PLAYS = 30                      # team-games with fewer runs and passes in the data are left out
TRAIN, TEST = (2006, 2019), (2020, 2025)

eff_points = (
    efficiency.filter(pl.col("plays") >= MIN_PLAYS)
    .with_columns(success_rate=pl.col("successes") / pl.col("plays"), newer=pl.col("season") >= NEWER_FROM)
    .join(pl.concat([games.select("game_id", team_id="home_id", points="home_points"),
                     games.select("game_id", team_id="away_id", points="away_points")]).filter(pl.col("points").is_not_null()),
          on=["game_id", "team_id"], how="left")
)
DESERVED_INPUTS = ["successes", "plays", "success_rate"]


def deserved_points_map(seasons):
    """For each set of play-by-play files, how many points usually go with a team's plays and successful plays in a game.
    Learned only from the given seasons."""
    maps = {}
    for newer in (False, True):
        t = eff_points.filter((pl.col("newer") == newer) & pl.col("season").is_between(*seasons) & pl.col("points").is_not_null())
        if t.height < 1000:
            continue
        X = np.column_stack([t.select(DESERVED_INPUTS).to_numpy(), np.ones(t.height)])
        maps[newer], *_ = np.linalg.lstsq(X, t["points"].to_numpy().astype(float), rcond=None)
    return maps


def deserved_points(maps):
    """The points each team 'deserved' in each game with play-by-play: {(game_id, team_id): points}."""
    out = {}
    for newer, weights in maps.items():
        t = eff_points.filter(pl.col("newer") == newer)
        X = np.column_stack([t.select(DESERVED_INPUTS).to_numpy(), np.ones(t.height)])
        out.update(zip(zip(t["game_id"].to_list(), t["team_id"].to_list()), (X @ weights).tolist()))
    return out


def points_ratings(games, prior=None, deserved=None, k=0.15, regress=0.05, home_field=0.5, non_fbs_gap=12,
                   start_avg=27.0, avg_rate=0.002, prior_pull=0.8, blend=0.5):
    """Each team's scoring offense and defense, in points vs. the league average, from earlier games only.
    With preseason ratings, each season starts with every team pulled toward its preseason rating.
    With deserved points, each game counts as a blend of the real score and the points the teams deserved."""
    prior, deserved = prior or {}, deserved or {}
    offense, defense, avg, season, rows = {}, {}, start_avg, None, []
    for g in games.iter_rows(named=True):
        if g["season"] != season:                          # new season: shrink ratings slightly toward average...
            season = g["season"]
            offense = {t: v * (1 - regress) for t, v in offense.items()}
            defense = {t: v * (1 - regress) for t, v in defense.items()}
            for t in offense:                              # ...then pull each team's margin toward its preseason rating
                if (season, t) in prior:
                    gap = prior[(season, t)] - (offense[t] - defense[t])
                    offense[t] += prior_pull * gap / 2
                    defense[t] -= prior_pull * gap / 2
        h, a = g["home_id"], g["away_id"]
        for team, is_fbs in ((h, g["home_fbs"]), (a, g["away_fbs"])):
            if team not in offense:                        # newcomers start at their preseason rating, or well below average
                start = prior.get((season, team), 0.0 if is_fbs else -2 * non_fbs_gap)
                offense[team], defense[team] = start / 2, -start / 2
        edge = 0.0 if g["neutral_site"] else home_field
        exp_home = avg + edge + offense[h] + defense[a]
        exp_away = avg - edge + offense[a] + defense[h]
        rows.append({"game_id": g["game_id"], "exp_home_points": exp_home, "exp_away_points": exp_away, "league_avg": avg})
        if g["result"] is None:
            continue
        home_pts, away_pts = float(g["home_points"]), float(g["away_points"])
        deserved_home, deserved_away = deserved.get((g["game_id"], h)), deserved.get((g["game_id"], a))
        if blend and deserved_home is not None and deserved_away is not None:
            home_pts = (1 - blend) * home_pts + blend * deserved_home
            away_pts = (1 - blend) * away_pts + blend * deserved_away
        miss_home, miss_away = home_pts - exp_home, away_pts - exp_away
        offense[h] += k * miss_home
        defense[a] += k * miss_home
        offense[a] += k * miss_away
        defense[h] += k * miss_away
        avg += avg_rate * ((g["home_points"] + g["away_points"]) / 2 - avg)
    return pl.DataFrame(rows)


def version3_margin(seasons):
    """Version 3's expected margin for every game, with deserved points learned from the given seasons."""
    ratings = points_ratings(games, prior=prior_by_team, deserved=deserved_points(deserved_points_map(seasons)), **POINTS_SETTINGS)
    return ratings.select("game_id", points_margin=pl.col("exp_home_points") - pl.col("exp_away_points"))


# Version 2's ratings (real scores only) for the comparison in Step 8 and for the totals model
v2_points = points_ratings(games, prior=prior_by_team, **{**POINTS_SETTINGS, "blend": 0.0})
df = df.join(v2_points, on="game_id").with_columns(
    points_margin_v2=pl.col("exp_home_points") - pl.col("exp_away_points"),
    base_total=pl.col("exp_home_points") + pl.col("exp_away_points"),
)
# Version 3's ratings for the test, learning deserved points only from the training seasons
df = df.join(version3_margin(TRAIN), on="game_id")

example = deserved_points_map(TRAIN).get(True)
if example is not None:
    def usual_points(plays, rate):
        return float(np.array([plays * rate, plays, rate, 1.0]) @ example)
    print(f"Deserved points (learned from {TRAIN[0]}–{TRAIN[1]}): a team that runs 70 plays scores about "
          f"{usual_points(70, 0.35):.0f} points with 35% of them successful, {usual_points(70, 0.45):.0f} with 45% "
          f"and {usual_points(70, 0.55):.0f} with 55%.")
else:
    print("Not enough play-by-play data for deserved points, so version 3's ratings match version 2's.")

## Step 7: The model

The model predicts the **final margin** (home score minus away score) from the same three inputs as before:

| Input | What it means |
|---|---|
| `elo_pts` | Elo gap between the teams, in points |
| `home` | 1 for a true home game, 0 at a neutral site (most bowl games) |
| `points_margin` | Expected margin from the points ratings, which now learn from deserved points too |

It's a linear regression, and the win probability comes from the predicted margin and how far real margins typically land from it (about 16 points in college).

In [ ]:
FEATURES = ["elo_pts", "home", "points_margin"]
FEATURES_V2 = ["elo_pts", "home", "points_margin_v2"]   # version 2, for comparison

data = df.with_columns(
    elo_pts=(pl.col("elo_home") - pl.col("elo_away")) / ELO_PER_POINT,
    home=(~pl.col("neutral_site")).cast(pl.Float64),
)


def fit_margin_model(data, seasons, features=FEATURES):
    """Learn one weight per input from the given seasons. Returns the weights and the typical miss (sigma)."""
    train = data.filter(pl.col("season").is_between(*seasons) & pl.col("result").is_not_null())
    X = train.select(features).to_numpy()
    y = train["result"].to_numpy().astype(float)
    weights, *_ = np.linalg.lstsq(X, y, rcond=None)
    sigma = float(np.std(y - X @ weights))
    return weights, sigma


def predict(data, weights, sigma, features=FEATURES):
    margin = data.select(features).to_numpy() @ weights
    return data.with_columns(
        model_spread=pl.Series(margin),                       # positive = home team favored
        model_p_home=pl.Series(norm.cdf(margin / sigma)),     # chance the home team wins
    )


def log_loss(p, y):
    p = p.clip(1e-12, 1 - 1e-12)
    return float(-(y * p.log() + (1 - y) * (1 - p).log()).mean())


def brier(p, y):
    return float(((p - y) ** 2).mean())


def accuracy(p, y):
    return float(((p > 0.5) == (y == 1)).mean())


def decided(df):
    """Games with a final score that didn't end in a tie."""
    return df.filter(pl.col("result").is_not_null() & (pl.col("result") != 0)).with_columns(
        home_won=(pl.col("result") > 0).cast(pl.Float64)
    )

## Step 8: The final test, 2020–2025

Both versions are trained once on 2006–2019, then scored on 2020–2025. Version 3 learns its deserved points from 2006–2019 too, so nothing from the test seasons leaks in. Lower log loss and Brier are better; a coin flip scores 0.693 and 0.25. The first tables use games with sportsbook moneylines, which the data doesn't have for 2020. Against the spread, each model takes its side whenever its spread differs from the market's; break-even at −110 is **52.4%**.

The last tables show the spread miss for each season, and split it into each team's **first 3 games** and the rest of the season.

In [ ]:
weights, sigma = fit_margin_model(data, TRAIN)
weights_v2, sigma_v2 = fit_margin_model(data, TRAIN, FEATURES_V2)
v2 = predict(data, weights_v2, sigma_v2, FEATURES_V2).select("game_id", v2_spread="model_spread", v2_p_home="model_p_home")
scored = predict(data, weights, sigma).join(v2, on="game_id")

# Each team's games played so far this season, to look separately at the first weeks
played_before = (
    pl.concat([scored.select("game_id", "season", "kickoff", team="home_id"), scored.select("game_id", "season", "kickoff", team="away_id")])
    .sort("kickoff", "game_id").with_columns(n=pl.col("game_id").cum_count().over("season", "team") - 1)
    .group_by("game_id").agg(games_before=pl.col("n").min())
)
scored = scored.join(played_before, on="game_id").with_columns(
    phase=pl.when(pl.col("games_before") < 3).then(pl.lit("first 3 games")).otherwise(pl.lit("rest of season")))

test = decided(scored.filter(pl.col("season").is_between(*TEST) & pl.col("mkt_p_home").is_not_null()))
y, n = test["home_won"], test.height


def score_row(name, p, show_accuracy=True):
    return {"model": name, "games": n, "log_loss": round(log_loss(p, y), 4), "brier": round(brier(p, y), 4),
            "accuracy": round(accuracy(p, y), 3) if show_accuracy else None}


print("Games with sportsbook moneylines, 2020–2025:")
display(pl.DataFrame([
    score_row("Coin flip (50%)", pl.Series([0.5] * n), show_accuracy=False),
    score_row("Elo", test["elo_p_home"]),
    score_row("Version 2", test["v2_p_home"]),
    score_row("Version 3 (+ efficiency)", test["model_p_home"]),
    score_row("Betting market (vig removed)", test["mkt_p_home"]),
]))


def season_ll(col):
    return (-(pl.when(pl.col("home_won") == 1).then(pl.col(col).log()).otherwise((1 - pl.col(col)).log()))).mean().round(4)


print("Log loss by season:")
display(test.group_by("season").agg(
    pl.len().alias("games"),
    season_ll("v2_p_home").alias("version_2"),
    season_ll("model_p_home").alias("version_3"),
    season_ll("mkt_p_home").alias("market"),
).sort("season"))

LABELS = ["50–60%", "60–70%", "70–80%", "80–90%", "90%+"]
fav = test.with_columns(
    p_fav=pl.max_horizontal(pl.col("model_p_home"), 1 - pl.col("model_p_home")),
    fav_won=pl.when(pl.col("model_p_home") >= 0.5).then(pl.col("home_won")).otherwise(1 - pl.col("home_won")),
).with_columns(
    confidence=pl.when(pl.col("p_fav") <= 0.6).then(pl.lit(LABELS[0]))
    .when(pl.col("p_fav") <= 0.7).then(pl.lit(LABELS[1]))
    .when(pl.col("p_fav") <= 0.8).then(pl.lit(LABELS[2]))
    .when(pl.col("p_fav") <= 0.9).then(pl.lit(LABELS[3]))
    .otherwise(pl.lit(LABELS[4]))
)
print("Calibration (version 3):")
display(pl.DataFrame({"confidence": LABELS}).join(
    fav.group_by("confidence").agg(
        pl.len().alias("games"),
        pl.col("p_fav").mean().round(3).alias("predicted"),
        pl.col("fav_won").mean().round(3).alias("actual"),
    ),
    on="confidence", how="left",
))

spreads = scored.filter(pl.col("season").is_between(*TEST) & pl.col("result").is_not_null() & pl.col("spread_line").is_not_null())


def spread_row(name, col, games=None):
    games = spreads if games is None else games
    miss = games.select((pl.col(col) - pl.col("result")).abs().mean()).item()
    if col == "spread_line":
        return {"model": name, "average_miss": round(miss, 2)}
    bets = games.with_columns(
        gap=pl.col(col) - pl.col("spread_line"),            # positive = model likes the home team more than the market does
        vs_line=pl.col("result") - pl.col("spread_line"),   # positive = home team covered
    ).filter((pl.col("gap") != 0) & (pl.col("vs_line") != 0)).with_columns(won=(pl.col("gap") > 0) == (pl.col("vs_line") > 0))
    return {"model": name, "average_miss": round(miss, 2), "ats_win_rate": round(bets["won"].mean(), 3)}


print(f"Spreads ({spreads.height:,} games with a line):")
display(pl.DataFrame([
    spread_row("Elo", "elo_spread"),
    spread_row("Version 2", "v2_spread"),
    spread_row("Version 3", "model_spread"),
    spread_row("Betting market", "spread_line"),
]))

print("Average spread miss by season:")
display(spreads.group_by("season").agg(
    pl.len().alias("games"),
    (pl.col("v2_spread") - pl.col("result")).abs().mean().round(2).alias("version_2"),
    (pl.col("model_spread") - pl.col("result")).abs().mean().round(2).alias("version_3"),
    (pl.col("spread_line") - pl.col("result")).abs().mean().round(2).alias("market"),
).sort("season"))

print("Average spread miss, early season vs. the rest:")
pl.DataFrame([
    {"phase": phase, "games": g.height,
     "version_2": round(g.select((pl.col("v2_spread") - pl.col("result")).abs().mean()).item(), 2),
     "version_3": round(g.select((pl.col("model_spread") - pl.col("result")).abs().mean()).item(), 2),
     "market": round(g.select((pl.col("spread_line") - pl.col("result")).abs().mean()).item(), 2)}
    for phase in ("first 3 games", "rest of season") for g in [spreads.filter(pl.col("phase") == phase)]
])

### What the results mean

- **Version 3 is a little better than version 2 again.** Log loss drops from 0.549 to 0.548, and it picks the winner 71.6% of the time, up from 71.2%. Log loss is better in 3 of the 5 seasons with moneylines.
- **Spreads improve more steadily.** Version 3 misses by 12.87 points, versus 12.93 for version 2. It's closer in 5 of the 6 test seasons and about even in 2025, and it helps both early and late in the season.
- **The market is still clearly better**, with log loss of 0.538 and a 12.2-point spread miss. The gap is still biggest in teams' first 3 games: about 12.4 points for the market and 13.6 for the model.
- **It isn't a betting edge.** Against the spread it wins 49.9% of its bets, below the 52.4% needed to profit.

## Step 9: The totals model (over/unders)

The totals model predicts **combined points** from the points ratings' expected total, with a weight and a constant learned from past games. For the over/under it then checks how far its total is from the betting line, and how often games with that gap went over in past seasons. That keeps its over chance from being overconfident. It uses version 2's points ratings, because deserved points didn't make totals any better.

**League average only** is the simplest possible guess, the same total for every game.

In [ ]:
from sklearn.linear_model import LogisticRegression

TOTAL_FEATURES = ["base_total"]


def fit_total_model(data, seasons):
    """Learn a weight for the points ratings' expected total, plus a constant."""
    train = data.filter(pl.col("season").is_between(*seasons) & pl.col("total").is_not_null())
    X = np.column_stack([train.select(TOTAL_FEATURES).to_numpy(), np.ones(train.height)])
    weights, *_ = np.linalg.lstsq(X, train["total"].to_numpy().astype(float), rcond=None)
    return weights


def predict_total(data, weights):
    X = np.column_stack([data.select(TOTAL_FEATURES).to_numpy(), np.ones(data.height)])
    return data.with_columns(model_total=pl.Series(X @ weights))


def fit_over_chance(scored, seasons):
    """How often games went over, given how far the model's total was from the betting line."""
    t = scored.filter(
        pl.col("season").is_between(*seasons) & pl.col("total").is_not_null()
        & pl.col("total_line").is_not_null() & (pl.col("total") != pl.col("total_line"))
    )
    gap = (t["model_total"] - t["total_line"]).to_numpy().reshape(-1, 1)
    return LogisticRegression(C=1e6).fit(gap, (t["total"] > t["total_line"]).to_numpy().astype(int))


def add_over_chance(scored, over_model):
    line = scored["total_line"].to_numpy()
    gap = np.nan_to_num(scored["model_total"].to_numpy() - line).reshape(-1, 1)
    p = over_model.predict_proba(gap)[:, 1]
    p[np.isnan(line)] = np.nan
    return scored.with_columns(model_p_over=pl.Series(p, nan_to_null=True))


total_weights = fit_total_model(data, TRAIN)
tscored = predict_total(data, total_weights)
tscored = add_over_chance(tscored, fit_over_chance(tscored, TRAIN))
ttest = tscored.filter(pl.col("season").is_between(*TEST) & pl.col("total").is_not_null() & pl.col("total_line").is_not_null())


def average_miss(expr):
    return round(ttest.select((expr - pl.col("total")).abs().mean()).item(), 2)


display(pl.DataFrame([
    {"prediction": "League average only", "games": ttest.height, "average_miss": average_miss(2 * pl.col("league_avg"))},
    {"prediction": "Totals model", "games": ttest.height, "average_miss": average_miss(pl.col("model_total"))},
    {"prediction": "Betting market line", "games": ttest.height, "average_miss": average_miss(pl.col("total_line"))},
]))

ou = ttest.filter(pl.col("total") != pl.col("total_line")).with_columns(
    went_over=(pl.col("total") > pl.col("total_line")).cast(pl.Float64),
    gap=pl.col("model_total") - pl.col("total_line"),            # positive = model leans over
)
bets = ou.filter(pl.col("gap") != 0).with_columns(won=(pl.col("gap") > 0) == (pl.col("went_over") == 1))
big = bets.filter(pl.col("gap").abs() >= 3)
display(pl.DataFrame([
    {"measure": "Over/under picks won", "value": f"{bets['won'].mean():.1%} of {bets.height}"},
    {"measure": "...when 3+ points from the line", "value": f"{big['won'].mean():.1%} of {big.height}"},
    {"measure": "Over chance log loss: model", "value": f"{log_loss(ou['model_p_over'], ou['went_over']):.4f}"},
    {"measure": "Over chance log loss: always 50%", "value": f"{log_loss(pl.Series([0.5] * ou.height), ou['went_over']):.4f}"},
]))

ttest.group_by("season").agg(
    pl.len().alias("games"),
    (pl.col("model_total") - pl.col("total")).abs().mean().round(2).alias("model_miss"),
    (pl.col("total_line") - pl.col("total")).abs().mean().round(2).alias("market_miss"),
).sort("season")

### What the totals results mean

- **It predicts scoring better than a league-average guess, but the market does better still.** The model misses the real total by about 13.0 points, the market's line by about 12.6, and a league-average guess by about 13.6. The market is closer in every test season.
- **Its over/under leans are a coin flip.** They win about half the time, below the 52.4% needed to profit, so its over chance stays close to 50%.
- **It still earns its place in the app.** Paired with the margin model, it gives every game a projected final score.

## Step 10: This week's games

For live picks, both models are retrained on every complete season from 2006 on, and deserved points are relearned from every complete season too.

**Projected score:** the home team's points are half the total plus half the margin, and the away team's are half the total minus half the margin.

There are about 60 games a week, so the list is long. To see just some of them, list teams in `SHOW_TEAMS` or conferences in `SHOW_CONFERENCES`, spelled the way they appear in the table. The conference names are `ACC`, `American Athletic`, `Big 12`, `Big Ten`, `Conference USA`, `FBS Independents`, `Mid-American`, `Mountain West`, `Pac-12`, `SEC` and `Sun Belt`.

**Reading the game cards**
- **Model**: who the model expects to win, and by how much. **Vegas**: the sportsbook spread, written the usual way, where the favorite gets a minus sign. *Memphis -7* means the sportsbooks have Memphis favored by 7.
- **Win chance**: the favorite's chance to win according to the model, with the sportsbooks' chance in parentheses.
- **Projected score** and **Total**: the model's expected score and combined points, next to the sportsbook over/under. Scores show one decimal so they always add up to the total and differ by the spread.
- **Model vs Vegas**: which side of the sportsbook lines the model leans to, and by how many points. Leans are usually small, and the tests show they don't beat the sportsbooks, so they're not betting advice.

At the top, a short list shows the games where the model and Vegas disagree most. To also see the full table of numbers, set `SHOW_NUMBERS_TABLE = True` in the cell below.

In [ ]:
# Teams or conferences to show. Leave both empty for every game.
# Examples: SHOW_TEAMS = ["Alabama", "Ohio State"]   SHOW_CONFERENCES = ["SEC", "Big Ten"]
SHOW_TEAMS = []
SHOW_CONFERENCES = []

SHOW_NUMBERS_TABLE = False   # True = also show the full table of numbers under the game cards


def signed(x):
    return "PK" if x == 0 else f"{x:+g}"


def game_card(g, when, home, away, extra=""):
    """One game in plain words. Spreads are stored from the home team's side; here they're written the way sportsbooks do."""
    ms, ml, mt, tl = g["model_spread"], g["spread_line"], g["model_total"], g["total_line"]
    pick = f"{home if ms > 0 else away} by {abs(ms):.1f}" if abs(ms) >= 0.05 else "even"
    vegas = "no line yet" if ml is None else "pick'em" if ml == 0 else f"{home if ml > 0 else away} {-abs(ml):g}"
    p = g["model_p_home"]
    fav, chance = (home, p) if p >= 0.5 else (away, 1 - p)
    win = f"{fav} {chance:.0%}"
    if g["mkt_p_home"] is not None:
        win += f" (Vegas {g['mkt_p_home'] if fav == home else 1 - g['mkt_p_home']:.0%})"
    lines = [f"{when} · {away} at {home}",
             f"  Model: {pick}   Vegas: {vegas}",
             f"  Win chance: {win}",
             f"  Projected score: {away} {(mt - ms) / 2:.1f}, {home} {(mt + ms) / 2:.1f}   "
             f"Total: {mt:.1f}" + (f" (Vegas {tl:g})" if tl is not None else "")]
    leans = []
    if ml is not None and abs(ms - ml) >= 0.05:
        side = home if ms > ml else away
        leans.append(f"{side} {signed(-ml if side == home else ml)} by {abs(ms - ml):.1f}")
    if tl is not None and abs(mt - tl) >= 0.05:
        leans.append(f"{'Over' if mt > tl else 'Under'} {tl:g} by {abs(mt - tl):.1f}")
    if leans:
        lines.append("  Model vs Vegas: leans " + " and ".join(leans) + " points")
    if extra:
        lines.append("  " + extra)
    return "\n".join(lines)


def disagreements(week, home_col, away_col, n=5):
    """The games where the model and Vegas disagree most, on the spread and on the total."""
    w = week.filter(pl.col("spread_line").is_not_null()).with_columns(gap=(pl.col("model_spread") - pl.col("spread_line")).abs()).sort("gap", descending=True).head(n)
    t = week.filter(pl.col("total_line").is_not_null()).with_columns(gap=(pl.col("model_total") - pl.col("total_line")).abs()).sort("gap", descending=True).head(n)
    out = ["Where the model and Vegas disagree most. The tests show leans like these win only about half the time, "
           "below the 52.4% needed to profit, so they're not betting advice."]
    for g in w.iter_rows(named=True):
        side = g[home_col] if g["model_spread"] > g["spread_line"] else g[away_col]
        line = -g["spread_line"] if side == g[home_col] else g["spread_line"]
        out.append(f"  Spread: {g[away_col]} at {g[home_col]}: model leans {side} {signed(line)} by {g['gap']:.1f} points")
    for g in t.iter_rows(named=True):
        out.append(f"  Total: {g[away_col]} at {g[home_col]}: model leans {'Over' if g['model_total'] > g['total_line'] else 'Under'} "
                   f"{g['total_line']:g} by {g['gap']:.1f} points")
    return "\n".join(out)


LIVE_SEASONS = (2006, CURRENT_SEASON - 1)
# Version 3's ratings again, now learning deserved points from every complete season
live_data = data.drop("points_margin").join(version3_margin(LIVE_SEASONS), on="game_id")
live_weights, live_sigma = fit_margin_model(live_data, LIVE_SEASONS)
live_total_weights = fit_total_model(live_data, LIVE_SEASONS)
live = predict_total(predict(live_data, live_weights, live_sigma), live_total_weights)
live = add_over_chance(live, fit_over_chance(live, LIVE_SEASONS))

w = dict(zip(FEATURES, live_weights))
print(f"Trained on {LIVE_SEASONS[0]}–{LIVE_SEASONS[1]}. What each input is worth, in points:")
print(f"  {ELO_PER_POINT} points of Elo gap:              {w['elo_pts']:.2f}")
print(f"  Home field:                         {w['home']:.2f}")
print(f"  1 point of points-ratings margin:   {w['points_margin']:.2f}")
print(f"  Totals: 1 point of expected total:  {live_total_weights[0]:.2f}")

upcoming = live.filter(pl.col("result").is_null() & (pl.col("kickoff") > now_utc)).sort("kickoff", "game_id")
if upcoming.is_empty():
    print("\nNo upcoming games found.")
else:
    nxt = upcoming.row(0, named=True)
    week = upcoming.filter(
        (pl.col("season") == nxt["season"]) & (pl.col("week") == nxt["week"]) & (pl.col("season_type") == nxt["season_type"])
    ).with_columns(
        kickoff_et=pl.col("kickoff").dt.convert_time_zone("America/New_York").dt.strftime("%a %b %d %I:%M %p"),
        away_pts=((pl.col("model_total") - pl.col("model_spread")) / 2).round(1),
        home_pts=((pl.col("model_total") + pl.col("model_spread")) / 2).round(1),
    )
    show = week
    if SHOW_TEAMS or SHOW_CONFERENCES:
        show = week.filter(
            pl.col("home_team").is_in(SHOW_TEAMS) | pl.col("away_team").is_in(SHOW_TEAMS)
            | pl.col("home_conference").is_in(SHOW_CONFERENCES) | pl.col("away_conference").is_in(SHOW_CONFERENCES)
        )
    label = "postseason" if nxt["season_type"] == "postseason" else f"week {nxt['week']}"
    print(f"\n{nxt['season']} season, {label}: {week.height} games, {week['spread_line'].is_not_null().sum()} with betting lines. Times are Eastern.\n")
    print(disagreements(show, "home_team", "away_team") + "\n")
    for g in show.sort("kickoff", "game_id").iter_rows(named=True):
        when = g["kickoff"].astimezone(EASTERN).strftime("%a %b %-d, %-I:%M %p")
        print(game_card(g, when, g["home_team"], g["away_team"]) + "\n")
    if SHOW_NUMBERS_TABLE:
        with pl.Config(tbl_rows=-1, fmt_str_lengths=40):
            display(show.select(
                "kickoff_et", "away_team", "home_team",
                elo_spread=pl.col("elo_spread").round(1), model_spread=pl.col("model_spread").round(1), market_spread=pl.col("spread_line"),
                model_home_win=pl.col("model_p_home").round(3), market_home_win=pl.col("mkt_p_home").round(3),
                model_total=pl.col("model_total").round(1), market_total=pl.col("total_line"), model_over_chance=pl.col("model_p_over").round(3),
            ))

## Step 11: Save this week's picks

College picks go in the same log as before, `ncaa_pick_log.csv` in your Google Drive:

1. Saves every model's numbers for this week's games that have a betting line and haven't kicked off, with the time you ran it.
2. Keeps the first pick for each game and model. Running it again later never changes a logged pick.
3. Fills in final scores and grades each model: log loss against the market, a record against the spread and on totals, and the average miss on totals.

The margin model logs as `cfb_margin_v3`, the same as in version 3, next to the earlier versions' picks (`cfb_margin_v1`, `cfb_margin_v2`) already in the log. Elo and totals keep their names, and games they've already logged aren't logged again. The college data has no closing lines, so this log doesn't track line movement the way the NFL log does.

In [ ]:
from pathlib import Path

ELO_NAME = f"cfb_elo_v1 (k={ELO_SETTINGS['k']}, home_field={ELO_SETTINGS['home_field']}, regress={ELO_SETTINGS['regress']})"
MODELS = {
    # log name: {log column: column in `live`}
    ELO_NAME: {"model_p_home": "elo_p_home", "model_spread": "elo_spread"},
    "cfb_margin_v3": {"model_p_home": "model_p_home", "model_spread": "model_spread"},
    "cfb_totals_v1": {"model_total": "model_total", "model_p_over": "model_p_over"},
}
MODEL_COLUMNS = {"model_p_home": 3, "model_spread": 1, "model_total": 1, "model_p_over": 3}   # column: decimals

# In Colab the log goes to your Google Drive; anywhere else it goes next to this notebook
try:
    from google.colab import drive
    drive.mount("/content/drive")
    LOG_PATH = Path("/content/drive/MyDrive/ncaa_pick_log.csv")
except ImportError:
    LOG_PATH = Path("ncaa_pick_log.csv")

PICK_COLUMNS = {
    "logged_at_et": pl.String, "model": pl.String,
    "season": pl.Int32, "week": pl.Int32, "season_type": pl.String, "game_id": pl.Int64, "kickoff_et": pl.String,
    "away_team": pl.String, "home_team": pl.String,
    "model_p_home": pl.Float64, "market_p_home": pl.Float64,
    "model_spread": pl.Float64, "market_spread": pl.Float64,
    "model_total": pl.Float64, "market_total": pl.Float64, "model_p_over": pl.Float64,
    "home_moneyline": pl.Int64, "away_moneyline": pl.Int64,
}


def read_log():
    if not LOG_PATH.exists():
        return pl.DataFrame(schema=PICK_COLUMNS)
    log = pl.read_csv(LOG_PATH, infer_schema_length=0)       # read as text, then fix the types
    log = log.with_columns([pl.lit(None, pl.String).alias(c) for c in PICK_COLUMNS if c not in log.columns])
    return log.select([pl.col(c).cast(t) for c, t in PICK_COLUMNS.items()])


def next_week_picks(live, now):
    """Every model's numbers for next week's games that have a betting line and haven't kicked off."""
    upcoming = live.filter(pl.col("result").is_null() & pl.col("spread_line").is_not_null() & (pl.col("kickoff") > now))
    if upcoming.is_empty():
        return pl.DataFrame(schema=PICK_COLUMNS)
    first = upcoming.sort("kickoff").row(0, named=True)
    week = upcoming.filter(
        (pl.col("season") == first["season"]) & (pl.col("week") == first["week"]) & (pl.col("season_type") == first["season_type"])
    ).sort("kickoff", "game_id")
    picks = []
    for name, cols in MODELS.items():
        model_values = {c: (pl.col(cols[c]).round(d) if c in cols else pl.lit(None, pl.Float64)) for c, d in MODEL_COLUMNS.items()}
        picks.append(week.select(
            logged_at_et=pl.lit(now.astimezone(EASTERN).strftime("%Y-%m-%d %H:%M")),
            model=pl.lit(name),
            season=pl.col("season"), week=pl.col("week"), season_type=pl.col("season_type"), game_id=pl.col("game_id"),
            kickoff_et=pl.col("kickoff").dt.convert_time_zone("America/New_York").dt.strftime("%Y-%m-%d %H:%M"),
            away_team=pl.col("away_team"), home_team=pl.col("home_team"),
            market_p_home=pl.col("mkt_p_home").round(3),
            market_spread=pl.col("spread_line"),
            market_total=pl.col("total_line"),
            home_moneyline=pl.col("home_moneyline"), away_moneyline=pl.col("away_moneyline"),
            **model_values,
        ).select(list(PICK_COLUMNS)).cast(PICK_COLUMNS))
    return pl.concat(picks)


def update_log(live, now=None):
    """Add new pre-kickoff picks, fill in final scores, and save the log."""
    now = now or datetime.now(timezone.utc)
    log = read_log()
    # The first pick for each game and model is the one that counts: logged picks are never replaced
    added = next_week_picks(live, now).join(log, on=["game_id", "model"], how="anti")
    log = pl.concat([log, added])
    finals = live.filter(pl.col("result").is_not_null()).select(
        "game_id", "away_points", "home_points", "result", total_points=pl.col("total"))
    log = log.join(finals, on="game_id", how="left").sort("kickoff_et", "model", "game_id")
    log.write_csv(LOG_PATH)
    return log, added


def log_scorecard(log):
    """Grade each model on its logged games that have final scores."""
    graded = log.filter(pl.col("result").is_not_null())
    if graded.is_empty():
        print("No logged games have finished yet. Run this again after the week's games to grade them.")
        return
    print(f"Graded picks from {graded['kickoff_et'].min()[:10]} to {graded['kickoff_et'].max()[:10]}. Break-even at -110 is 52.4%.\n")

    sides = graded.filter(pl.col("model_p_home").is_not_null()).with_columns(
        home_won=pl.when(pl.col("result") > 0).then(1.0).when(pl.col("result") < 0).then(0.0).otherwise(0.5),
        gap=pl.col("model_spread") - pl.col("market_spread"),       # positive = model likes the home team more
        vs_line=pl.col("result") - pl.col("market_spread"),         # positive = home team covered
    )
    totals = graded.filter(pl.col("model_total").is_not_null() & pl.col("market_total").is_not_null()).with_columns(
        gap=pl.col("model_total") - pl.col("market_total"),         # positive = model leans over
        vs_line=pl.col("total_points") - pl.col("market_total"),    # positive = went over
    )

    def record(g):
        bets = g.filter(pl.col("gap") != 0)
        pushes = bets.filter(pl.col("vs_line") == 0).height
        wins = bets.filter((pl.col("vs_line") != 0) & ((pl.col("gap") > 0) == (pl.col("vs_line") > 0))).height
        losses = bets.height - wins - pushes
        return {"record": f"{wins}-{losses}-{pushes}", "win_rate": round(wins / (wins + losses), 3) if wins + losses else None}

    if not sides.is_empty():
        print("Winners and spreads:")
        rows = []
        for name in sides["model"].unique(maintain_order=True):
            g = sides.filter(pl.col("model") == name)
            m = g.filter(pl.col("market_p_home").is_not_null())
            rows.append({"model": name, "graded_games": g.height,
                         "model_log_loss": round(log_loss(m["model_p_home"], m["home_won"]), 4) if m.height else None,
                         "market_log_loss": round(log_loss(m["market_p_home"], m["home_won"]), 4) if m.height else None,
                         **record(g)})
        display(pl.DataFrame(rows))
    if not totals.is_empty():
        print("Totals:")
        display(pl.DataFrame([
            {"model": name, "graded_games": g.height,
             "model_miss": round(g.select((pl.col("model_total") - pl.col("total_points")).abs().mean()).item(), 2),
             "market_miss": round(g.select((pl.col("market_total") - pl.col("total_points")).abs().mean()).item(), 2),
             **record(g)}
            for name in totals["model"].unique(maintain_order=True)
            for g in [totals.filter(pl.col("model") == name)]
        ]))

In [ ]:
log, added = update_log(live)
print(f"Pick log: {LOG_PATH} ({log.height} picks logged so far)\n")

if added.is_empty():
    print("Nothing new to log. This week's games are already in the log, or none have betting lines yet.\n")
else:
    for name in added["model"].unique(maintain_order=True):
        a = added.filter(pl.col("model") == name)
        print(f"Logged {a.height} games for {name}, {a['season'][0]} season, week {a['week'][0]}")
    print()

log_scorecard(log)

## Step 12: Save the website data

The picks website reads one file per league. This step writes `app_data/cfb.json` in your Google Drive, next to your pick log. It holds:
- **This week's games**: every game with an FBS team, the model's numbers, the sportsbooks' numbers where there's a line, each team's conference, and the final score for games that are over.
- **What the Top picks lists need**: each game's chance to cover the spread, found the same way Step 9 finds the over chance. The repository's `scripts/top_picks.py` builds the website's Top picks lists from it.
- **How the model is doing**: the season record and every graded game this season. They're graded from the pick log, the same way Step 11 grades them. When a game was picked by more than one version of the margin model, the newest version's pick counts, the same rule `game_review_v1` uses. When `game_review_v1` has reviewed a game, its explanation comes along too.

- **The report card**: how accurate the picks have been on the model's own terms: winners picked, the average miss on the margin and the total, and whether its win chances came true. Next to each is the same number from the 2020–2025 test seasons in Step 8, so you can see whether this season is better or worse than usual. The sportsbooks' numbers are included only for reference.

Spreads in the file are written from the home team's side: a `home_margin` of -7 means the away team is expected to win by 7. The website turns that into sportsbook style, like *Memphis -7*. Kickoff times include their time zone, so the website shows them in each visitor's own time.

**The website itself** is `index.html` in the same `app_data` folder. Once it's online, it reads `nfl.json` and `cfb.json` from its own folder. To look at it before then, download `index.html`, open it in your browser and drop the two `.json` files onto the page.

In [ ]:
import json

APP_FOLDER = LOG_PATH.parent / "app_data"           # the website's data, next to your pick log
APP_FILE = APP_FOLDER / "cfb.json"
REVIEW_FILE = LOG_PATH.parent / "ncaa_game_reviews.csv"   # written by game_review_v1
ELO_PREFIX = "cfb_elo_v"                               # Elo is logged but isn't the main model, as in the review


def fit_cover_chance(scored, seasons):
    """How often the home team covered, given how far the model's spread was from the betting line. Built like the over chance in Step 9."""
    t = scored.filter(
        pl.col("season").is_between(*seasons) & pl.col("result").is_not_null()
        & pl.col("spread_line").is_not_null() & (pl.col("result") != pl.col("spread_line"))
    )
    gap = (t["model_spread"] - t["spread_line"]).to_numpy().reshape(-1, 1)
    return LogisticRegression(C=1e6).fit(gap, (t["result"] > t["spread_line"]).to_numpy().astype(int))


cover_model = fit_cover_chance(live, LIVE_SEASONS)


def cover_chance(g):
    """The model's chance that the home team covers the sportsbook spread. The Top picks lists use it."""
    if g["spread_line"] is None or g["model_spread"] is None:
        return None
    return float(cover_model.predict_proba([[g["model_spread"] - g["spread_line"]]])[0, 1])


def clean(value, digits=None):
    """A value ready for the website: numbers rounded, missing values as null."""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(value, float) and digits is not None:
        return round(value, digits)
    return value


def side_result(model_line, market_line, actual):
    """Win, loss or push for the side the model liked compared with the sportsbook line."""
    if model_line is None or market_line is None or actual is None:
        return None
    if model_line == market_line:
        return "no pick"
    if actual == market_line:
        return "push"
    return "win" if (model_line > market_line) == (actual > market_line) else "loss"


def logged_picks(log):
    """One row per logged game: the newest margin pick (Elo aside), the totals pick and the final score."""
    margin = (log.filter(pl.col("model_spread").is_not_null() & ~pl.col("model").str.starts_with(ELO_PREFIX))
              .sort("logged_at_et").group_by("game_id").agg(
                  margin_model=pl.col("model").last(), pick_margin=pl.col("model_spread").last(),
                  pick_win=pl.col("model_p_home").last(), line_margin=pl.col("market_spread").last()))
    totals = (log.filter(pl.col("model_total").is_not_null()).sort("logged_at_et").group_by("game_id").agg(
                  pick_total=pl.col("model_total").last(), line_total=pl.col("market_total").last()))
    games = log.sort("logged_at_et").group_by("game_id").agg(
        pl.col("season").last(), pl.col("week").last(), pl.col("kickoff_et").last(),
        pl.col("away_team").last(), pl.col("home_team").last(),
        away_final=pl.col("away_points").last(), home_final=pl.col("home_points").last())
    return games.join(margin, on="game_id", how="left").join(totals, on="game_id", how="left")


def graded_results(picks, season, reviews):
    """Every finished game this season that the models picked, graded the same way as the pick log."""
    out = []
    done = picks.filter((pl.col("season") == season) & pl.col("home_final").is_not_null()).sort("kickoff_et", "game_id", descending=True)
    for r in done.iter_rows(named=True):
        margin, total = r["home_final"] - r["away_final"], r["home_final"] + r["away_final"]
        pick = r["pick_margin"]
        review = reviews.get(str(r["game_id"]), {})
        out.append({
            "id": r["game_id"], "week": r["week"],
            "kickoff": datetime.strptime(r["kickoff_et"], "%Y-%m-%d %H:%M").replace(tzinfo=EASTERN).isoformat(),
            "away": r["away_team"], "home": r["home_team"],
            "final": {"away": r["away_final"], "home": r["home_final"]},
            "model": {"home_margin": clean(pick, 1), "home_win": clean(r["pick_win"], 3), "total": clean(r["pick_total"], 1)},
            "vegas": {"home_margin": clean(r["line_margin"]), "total": clean(r["line_total"])},
            "winner": None if pick is None or pick == 0 or margin == 0 else (pick > 0) == (margin > 0),
            "ats": side_result(pick, r["line_margin"], margin),
            "over_under": side_result(r["pick_total"], r["line_total"], total),
            "review": review.get("summary") or None,
            "total_review": review.get("total_summary") or None,
        })
    return out


def tally(results):
    """Season-style record for a list of graded games."""
    def count(key):
        values = [r[key] for r in results]
        return {"win": values.count("win"), "loss": values.count("loss"), "push": values.count("push")}

    picked = [r for r in results if r["winner"] is not None]
    spread_games = [r for r in results if r["model"]["home_margin"] is not None and r["vegas"]["home_margin"] is not None]
    total_games = [r for r in results if r["model"]["total"] is not None and r["vegas"]["total"] is not None]

    def miss(games, key, side):
        if not games:
            return None
        actual = (lambda g: g["final"]["home"] - g["final"]["away"]) if key == "home_margin" else (lambda g: g["final"]["home"] + g["final"]["away"])
        return round(sum(abs(actual(g) - g[side][key]) for g in games) / len(games), 1)

    return {
        "games": len(results),
        "winners": {"right": sum(r["winner"] for r in picked), "picked": len(picked)},
        "ats": count("ats"), "over_under": count("over_under"),
        "spread_miss": {"model": miss(spread_games, "home_margin", "model"), "vegas": miss(spread_games, "home_margin", "vegas"), "games": len(spread_games)},
        "total_miss": {"model": miss(total_games, "total", "model"), "vegas": miss(total_games, "total", "vegas"), "games": len(total_games)},
    }


def game_status(g, now):
    if g["result"] is not None:
        return "final"
    return "upcoming" if g["kickoff"] > now else "in_progress"


now = datetime.now(timezone.utc)
ahead = live.filter(pl.col("result").is_null() & (pl.col("kickoff") > now))
anchor = ahead.sort("kickoff").row(0, named=True) if ahead.height else live.filter(pl.col("result").is_not_null()).sort("kickoff").row(-1, named=True)
this_week = live.filter(
    (pl.col("season") == anchor["season"]) & (pl.col("week") == anchor["week"]) & (pl.col("season_type") == anchor["season_type"])
).sort("kickoff", "game_id")
label = "Bowl season" if anchor["season_type"] == "postseason" else f"Week {anchor['week']}"

games_out = []
for g in this_week.iter_rows(named=True):
    status = game_status(g, now)
    games_out.append({
        "id": g["game_id"], "kickoff": g["kickoff"].astimezone(EASTERN).isoformat(), "status": status,
        "away": g["away_team"], "home": g["home_team"],
        "away_conference": g["away_conference"], "home_conference": g["home_conference"],
        "neutral": bool(g["neutral_site"]),
        "fbs_vs_fbs": bool(g["home_fbs"] and g["away_fbs"]),     # False when one team is from a lower division
        "model": {"home_margin": clean(g["model_spread"], 1), "home_win": clean(g["model_p_home"], 3),
                  "total": clean(g["model_total"], 1), "over": clean(g["model_p_over"], 3),
                  "home_cover": clean(cover_chance(g), 3)},
        "vegas": {"home_margin": clean(g["spread_line"]), "home_win": clean(g["mkt_p_home"], 3),
                  "total": clean(g["total_line"]), "over": clean(g["mkt_p_over"], 3),
                  "home_moneyline": clean(g["home_moneyline"]), "away_moneyline": clean(g["away_moneyline"])},
        "final": {"away": g["away_points"], "home": g["home_points"]} if status == "final" else None,
    })

reviews = {}
if REVIEW_FILE.exists():
    reviews = {str(r["game_id"]): r for r in pl.read_csv(REVIEW_FILE, infer_schema_length=0).iter_rows(named=True)}
results = graded_results(logged_picks(log), anchor["season"], reviews)
weeks = sorted({r["week"] for r in results}, reverse=True)

# The report card: how accurate the picks were on the model's own terms, next to the out-of-sample test seasons
# from Step 8. The sportsbooks' numbers ride along only for reference.
CONFIDENCE = [(0.6, "50–60%"), (0.7, "60–70%"), (0.8, "70–80%"), (0.9, "80–90%"), (1.01, "90%+")]


def accuracy(rows):
    """Winners picked, how far off the margin and total were, and the Brier score of the win chances (0 is perfect,
    0.25 is a coin flip). Each row has the model's home win chance p, margin m and total t, the sportsbooks' margin
    vm and total vt, and the final margin and total."""
    def avg(values, digits):
        values = [v for v in values if v is not None]
        return round(sum(values) / len(values), digits) if values else None
    decided = [r for r in rows if r["p"] is not None and r["margin"] != 0]
    return {
        "games": len(rows),
        "winners": avg([float((r["p"] >= 0.5) == (r["margin"] > 0)) for r in decided], 3),
        "brier": avg([(r["p"] - float(r["margin"] > 0)) ** 2 for r in decided], 3),
        "margin_miss": avg([abs(r["m"] - r["margin"]) for r in rows if r["m"] is not None], 1),
        "total_miss": avg([abs(r["t"] - r["total"]) for r in rows if r["t"] is not None], 1),
        "vegas_margin_miss": avg([abs(r["vm"] - r["margin"]) for r in rows if r["vm"] is not None and r["m"] is not None], 1),
        "vegas_total_miss": avg([abs(r["vt"] - r["total"]) for r in rows if r["vt"] is not None and r["t"] is not None], 1),
    }


def calibration(rows):
    """For each level of confidence: how many games, the chance the model gave its pick on average, and how often it won."""
    groups = {label: [] for _, label in CONFIDENCE}
    for r in rows:
        if r["p"] is None or r["margin"] == 0:
            continue
        fav = max(r["p"], 1 - r["p"])
        label = next(name for top, name in CONFIDENCE if fav <= top)
        groups[label].append((fav, float((r["p"] >= 0.5) == (r["margin"] > 0))))
    return [{"label": label, "games": len(g),
             "predicted": round(sum(f for f, _ in g) / len(g), 3) if g else None,
             "actual": round(sum(w for _, w in g) / len(g), 3) if g else None} for label, g in groups.items()]


bench = (
    scored.filter(pl.col("season").is_between(*TEST) & pl.col("result").is_not_null() & pl.col("spread_line").is_not_null())
    .select("game_id", "model_p_home", "model_spread", "spread_line", "result")
    .join(ttest.select("game_id", "model_total", "total_line", "total"), on="game_id", how="left")
)
bench_rows = [{"p": r["model_p_home"], "m": r["model_spread"], "t": r["model_total"], "vm": r["spread_line"], "vt": r["total_line"],
               "margin": r["result"], "total": r["total"]} for r in bench.iter_rows(named=True)]
live_rows = [{"week": r["week"], "p": r["model"]["home_win"], "m": r["model"]["home_margin"], "t": r["model"]["total"],
              "vm": r["vegas"]["home_margin"], "vt": r["vegas"]["total"],
              "margin": r["final"]["home"] - r["final"]["away"], "total": r["final"]["home"] + r["final"]["away"]} for r in results]
report = {
    "benchmark": {"seasons": f"{TEST[0]}–{TEST[1]}", **accuracy(bench_rows), "calibration": calibration(bench_rows)},
    "season": {**accuracy(live_rows), "calibration": calibration(live_rows)},
    "weeks": [{"week": w, **accuracy([r for r in live_rows if r["week"] == w])} for w in sorted({r["week"] for r in live_rows})],
}

app_data = {
    "league": "College",
    "updated": datetime.now(EASTERN).isoformat(timespec="minutes"),
    "season": anchor["season"], "week": anchor["week"], "week_label": label,
    "models": {"margin": "cfb_margin_v3", "totals": "cfb_totals_v1", "cover": "cover_chance_v1"},
    "games": games_out,
    "record": {"season": tally(results), "weeks": [{"week": w, **tally([r for r in results if r["week"] == w])} for w in weeks]},
    "results": results,
    "report": report,
}
APP_FOLDER.mkdir(parents=True, exist_ok=True)
APP_FILE.write_text(json.dumps(app_data, indent=1))

lined = sum(g["vegas"]["home_margin"] is not None for g in games_out)
reviewed = sum(r["review"] is not None for r in results)
print(f"Saved the website data to {APP_FILE}:")
print(f"  {len(games_out)} games for {anchor['season']} {label.lower()}, {lined} with betting lines and cover chances")
print(f"  {len(results)} graded games this season" + (f", {reviewed} with a post-game review" if results else ""))
b, s = report["benchmark"], report["season"]
print(f"  Report card: average margin miss {s['margin_miss']} this season vs. {b['margin_miss']} in the {b['seasons']} tests; "
      f"winners {s['winners']} vs. {b['winners']}" if s["games"] else f"  Report card: no graded games yet ({b['seasons']} tests: average margin miss {b['margin_miss']})")
if results and not REVIEW_FILE.exists():
    print(f"  No review file yet at {REVIEW_FILE}. Run game_review_v1 before this notebook to add the post-game explanations.")

## Next steps

The weekly runs, the website, the Top picks lists and the AI news check now happen in the `football-picks` repository. Its README explains them.

1. **Closing lines.** CollegeFootballData's betting lines for this season would let the college log track line movement, like the NFL log does.
2. **College player props.** The NFL props approach, using college box scores.